# Week 2 — Signal Conditioning Pipeline

We simulate the digital equivalent of an acquisition chain: anti-aliasing low-pass filtering, optional decimation, and 16-bit ADC quantization.

In [ ]:
import sys
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from dsp.signal_processor import SignalProcessor, ConditioningConfig

input_path = ROOT / 'data' / 'raw' / 'week1_synthetic_adc.h5'
with h5py.File(input_path, 'r') as h5:
    t = h5['time_s'][:]
    clean = h5['clean_v'][:]
    noisy = h5['noisy_v'][:]

cfg = ConditioningConfig()
processor = SignalProcessor(cfg)
result = processor.process(noisy)
filtered = result['filtered']
quantized = result['quantized']

print(f'Input sampling rate : {cfg.fs/1000:.1f} kHz')
print(f'LPF cutoff          : {cfg.cutoff_hz/1000:.1f} kHz')
print(f'Filter order        : {cfg.filter_order}')
print(f'ADC                 : {cfg.adc_bits}-bit, {cfg.adc_min:g} to {cfg.adc_max:g} V')

## 1. Anti-aliasing filter frequency response

In [ ]:
f, h = processor.filter_response()
mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
phase_deg = np.unwrap(np.angle(h)) * 180 / np.pi

plt.figure(figsize=(11, 4))
plt.plot(f/1000, mag_db)
plt.axvline(cfg.cutoff_hz/1000, linestyle='--', label='4 kHz cutoff')
plt.axvline(10, linestyle=':', label='10 kHz interference')
plt.xlim(0, 20)
plt.ylim(-100, 5)
plt.xlabel('Frequency [kHz]')
plt.ylabel('Magnitude [dB]')
plt.title('4th-Order Butterworth Anti-Aliasing Filter')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

idx = np.argmin(np.abs(f - cfg.cutoff_hz))
idx10 = np.argmin(np.abs(f - 10_000))
print(f'Gain at 4 kHz  : {mag_db[idx]:.2f} dB')
print(f'Gain at 10 kHz : {mag_db[idx10]:.2f} dB')

## 2. Before/after conditioning — Channel 2

In [ ]:
ch = 1
window = t < 0.010
plt.figure(figsize=(12, 5))
plt.plot(t[window]*1000, noisy[window,ch], label='Raw/noisy')
plt.plot(t[window]*1000, filtered[window,ch], label='After 4 kHz LPF')
plt.xlabel('Time [ms]')
plt.ylabel('Voltage [V]')
plt.title('CH2 Conditioning — Time Domain')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Frequency-domain effect

In [ ]:
def spectrum(x, fs):
    n = len(x)
    f = np.fft.rfftfreq(n, 1/fs)
    a = np.abs(np.fft.rfft(x))/n
    return f, 20*np.log10(np.maximum(a, 1e-12))

f0, raw_db = spectrum(noisy[:,ch], cfg.fs)
f1, filt_db = spectrum(filtered[:,ch], cfg.fs)
plt.figure(figsize=(11, 5))
plt.plot(f0/1000, raw_db, label='Raw')
plt.plot(f1/1000, filt_db, label='Filtered')
plt.xlim(0, 20)
plt.ylim(-100, 5)
plt.xlabel('Frequency [kHz]')
plt.ylabel('Magnitude [dB]')
plt.title('CH2 Spectrum Before vs After Anti-Aliasing Filter')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Optional decimation

The LPF must precede decimation. With factor 2, the effective sampling rate becomes 25 kHz and the new Nyquist frequency is 12.5 kHz.

In [ ]:
decimated = processor.downsample(filtered, factor=2)
print(f'Original samples : {len(filtered)}')
print(f'Decimated samples: {len(decimated)}')
print(f'Effective Fs     : {cfg.fs/2/1000:.1f} kHz')
print(f'New Nyquist      : {cfg.fs/4/1000:.1f} kHz')

## 5. Quantization check

In [ ]:
q_error = quantized - filtered
print(f'ADC LSB: {cfg.quantization_step*1e3:.5f} mV')
print(f'Quantization RMS on CH2: {np.sqrt(np.mean(q_error[:,ch]**2))*1e3:.5f} mV')